In [1]:
import json
import pandas as pd
from pprint import pprint

In [2]:
# Load unlabbled corpus
df_unlablled = pd.read_csv('/home/ameyh/mental-health-comorbitidy-classification/data/test/silver_labels_gpt.csv')
print(f"Total unlablled corpus: {df_unlablled.shape}")
df_unlablled.head(1)

Total unlablled corpus: (7667, 12)


,index,text,subreddit,author,system_description,prompt,gpt_prediction,Mental Health Disorder,Name of Mental Health Disorder,DSM5 Rationale,silver_label,id
0,11,"i woke up very early, 2 am. i just came out of...",ForeverAlone,SixViking,You are a Psychology professor working in the ...,"Reddit Post: ""i woke up very early, 2 am. i ju...",Mental Health Disorder: Yes\nName of Mental He...,Yes,Major Depressive Disorder (MDD),The language used in the post indicates severa...,Depression,4JEVyZ


In [3]:
# Load test set
df_test = pd.read_csv('/home/ameyh/mental-health-comorbitidy-classification/data/test/full_test.csv')
print(f"Total test set (blind): {df_test.shape}")
df_test.head(1)

Total test set (blind): (2872, 7)


,id,title,selftext,labels,disorder,text,groundtruth_label
0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression


In [4]:
mapping = json.load(open("/home/ameyh/mental-health-comorbitidy-classification/data/mappings/semantic-similarity-depression.json"))
print(f"Total mapping: {len(mapping)}")
pprint(mapping)

Total mapping: 2872
{'3zf986': [['NBV7BY', 0.660976231098175],
            ['4WH6NR', 0.6449497938156128],
            ['3MpoEm', 0.6357589960098267],
            ['4UUZ86', 0.6193178296089172],
            ['43x8aJ', 0.6063274145126343],
            ['xorcU5', 0.5969784259796143],
            ['YrVXRu', 0.5959591269493103],
            ['noYHmg', 0.5952088832855225],
            ['489p8S', 0.5915932655334473],
            ['3oy9Zt', 0.5890166759490967],
            ['44J6or', 0.58294677734375],
            ['4PL9dr', 0.5821625590324402],
            ['3HWqd2', 0.5807008743286133],
            ['3tBZbd', 0.5753974914550781],
            ['44GdU5', 0.5747628211975098],
            ['37C5kK', 0.5729663372039795],
            ['43rqkt', 0.5722278952598572],
            ['3r6KrN', 0.5708215832710266],
            ['omLtQm', 0.5663670301437378],
            ['32CxkC', 0.565447986125946],
            ['4VsXLZ', 0.56522136926651],
            ['355jG5', 0.5644267201423645],
            ['4Jo2

In [5]:
assert len(mapping) == df_test.shape[0]

#### Get top-k exemplars for a particular datapoint in test set 

In [6]:
sample_datapoint = df_test.sample()
sample_datapoint

,id,title,selftext,labels,disorder,text,groundtruth_label
2245,7m4m6j,My brother died and it hurts so much,My younger brother was found dead in his apart...,"[1, 0]",{'depressive_disorder'},My brother died and it hurts so much. My young...,Depression


In [7]:
exemplars = mapping[sample_datapoint.iloc[0]['id']]
pprint(exemplars)

[['rCn73f', 0.7188562154769897],
 ['3ybHCD', 0.7154527902603149],
 ['BKVBJK', 0.7026964426040649],
 ['4Fvdcu', 0.7014690637588501],
 ['3KrZay', 0.700445294380188],
 ['siihXv', 0.6891827583312988],
 ['3CzxnM', 0.6487476229667664],
 ['3dUzsu', 0.6467969417572021],
 ['WfkKBw', 0.6390045285224915],
 ['3NBxDC', 0.6388687491416931],
 ['KzmSLG', 0.6349929571151733],
 ['3LBG3Q', 0.6284743547439575],
 ['7UywVH', 0.6118215322494507],
 ['4DiPtP', 0.6038773059844971],
 ['xxtHMz', 0.6002994179725647],
 ['3hoNFK', 0.5984659194946289],
 ['4RM53c', 0.5964802503585815],
 ['eLNcar', 0.5786051154136658],
 ['3dUo3q', 0.5779073238372803],
 ['4JmaVk', 0.5759375095367432],
 ['wGqbmx', 0.5753626823425293],
 ['3VimP8', 0.5749728679656982],
 ['37Rpxx', 0.5731668472290039],
 ['pfYhsr', 0.5695798397064209],
 ['HAQDhb', 0.567937970161438],
 ['3KdbHZ', 0.5667355060577393],
 ['3aKuMw', 0.5627630352973938],
 ['3si6rs', 0.5580594539642334],
 ['JuEJqL', 0.5569185614585876],
 ['dSc4hS', 0.5567352771759033]]


In [8]:
exemplar_ids = [i[0] for i in exemplars]
exemplar_df = df_unlablled[df_unlablled['id'].isin(exemplar_ids)]
print(exemplar_df.shape)
exemplar_df.head(1)

(30, 12)


,index,text,subreddit,author,system_description,prompt,gpt_prediction,Mental Health Disorder,Name of Mental Health Disorder,DSM5 Rationale,silver_label,id
9,77,"hi there, i am currently at a very stressful t...",depression_help,Vikturus22,You are a Psychology professor working in the ...,"Reddit Post: ""hi there, i am currently at a ve...",Mental Health Disorder: Yes\n\nName of Mental ...,Yes,Major Depressive Disorder,The language and content of the Reddit post in...,Depression,3NBxDC


### Add exemplar as a column in df_test

In [10]:
df_test['exemplars_semantic_similarity'] = None
df_test['mapping_semantic_similarity'] = None
topk = 10

for i in range(df_test.shape[0]):
    exemplars = mapping[df_test.iloc[i]['id']]
    exemplar_ids = [i[0] for i in exemplars][:topk]
    exemplar_df = df_unlablled[df_unlablled['id'].isin(exemplar_ids)]
    exemplar_posts = exemplar_df['text'].values.tolist()
    df_test.at[i, 'exemplars_semantic_similarity'] = json.dumps(exemplar_posts)
    df_test.at[i, 'mapping_semantic_similarity'] = json.dumps(exemplars)

In [11]:
df_test.head(1)

,id,title,selftext,labels,disorder,text,groundtruth_label,exemplars_semantic_similarity,mapping_semantic_similarity
0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,"[""so i was going through my stuff since i was ...","[[""5jPx8U"", 0.7506369352340698], [""3ng2N5"", 0...."
